# Content Creation at Scale

In [1]:
import sys
import os
import yaml
from crewai import Agent, Task, Crew

# Warning control
import warnings
warnings.filterwarnings('ignore')

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you acan import your config
from config import api_key, serper_api_key, trello_api_key, trello_api_token, trello_board_id

import os

os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["OPENAI_API_KEY"] = api_key
os.environ["SERPER_API_KEY"] = serper_api_key

## Creating Structured Output

In [2]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, LinkedIn).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="The article, formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'agents': 'config5/agents.yaml',
    'tasks': 'config5/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [4]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


## Setup Multi LLM models
> Note: The 'llama-3.1-70b-versatil' model was deprecated. As a result, we updated it to the 'llama-3.3-70b-versatil' model in January 2025.

In [25]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-3.5-turbo'

# here the course uses a different llm
# groq_llm = "groq/llama-3.3-70b-versatile"
groq_llm = "gpt-4.1"

## Creating Crew, Agents, and Tasks

In [26]:
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=groq_llm,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=groq_llm,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()]
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai_tools/tools/base_tool.py:93: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  for arg, attribute in self.args_schema.schema()["properties"].items():
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/agent.py:397: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  args_schema = str(tool.model_fields)
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/alembic/config.py:598: DeprecationWarning: No path_separator found in configuration; falling back to 

## Kicking off the Crew

In [32]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflation lookout in the Netherlands for Q4 2025 and 2026'
})

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/agent.py:397: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  args_schema = str(tool.model_fields)
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/agent.py:397: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  args_schema = str(tool.model_fields)


# Agent: Lead Market Analyst
## Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular focus on Inflation lookout in the Netherlands for Q4 2025 and 2026. Identify and summarize the most relevant and impactful news items that could influence market trends or investor decisions. Utilize financial news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends, regulatory changes, or significant economic indicators that directly relate to Inflation lookout in the Netherlands for Q4 2025 and 2026.



/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/tools/tool_usage.py:161: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/tools/tool_usage.py:161: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute

 

I encountered an error while trying to use the tool. This was the error: 1 validation error for SerperDevToolSchema
search_query
  Field required [type=missing, input_value={'name': 'Netherlands inf... for Q4 2025 and 2026.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing.
 Tool Search the internet accepts these inputs: Search the internet(search_query: 'string') - A tool that can be used to search the internet with a search_query. search_query: 'Mandatory search query you want to use to search the internet'



# Agent: Lead Market Analyst
## Thought: Thought: To create a highly relevant and up-to-date summary report on the Inflation lookout in the Netherlands for Q4 2025 and 2026, I need to search for the latest financial news, expert outlooks, economic forecasts, and regulatory updates specifically targeting Dutch inflation projections for these periods. I will begin with a targeted search for such news and analysis.
## Using tool: S

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/tools/tool_usage.py:161: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"


...



# Agent: Chief Data Strategist
## Thought: Thought: The search revealed several useful PDFs and dedicated pages from European Commission and CPB (the Dutch Bureau for Economic Policy Analysis) that are likely to include visualizations, sentiment analysis, and sectoral observations. The European Economic Forecast Spring 2025 PDF was specifically cited as containing Q4 inflation data and projections into 2026. I should target that document for explicit visualizations and granular data.
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"Netherlands inflation forecast 2025 2026 visualization sector risk sentiment\", \"website\": \"https://economy-finance.ec.europa.eu/document/download/e9de23c8-b161-40d0-9ad7-e04a25500023_en?filename=ip318_en.pdf\"}"
## Tool Output: 
Relevant Content:
�:[�x?�V[+�][j�Q�³�?�pCgB8�P�';8�c�7�B�%�"Ш:�b�S�6nCjGg�[��(q�1� hшZPZ�m�t�v(�-��DE%�e�� Ç�!M�5OߴD�$�{�}羊 ͅӌ��=�|�8`�F>stream H�j�@�}�l/�4 �д`t� �*s� Oi�Y?�1_o7�Ѝ&

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/agent.py:397: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  args_schema = str(tool.model_fields)
/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/tools/tool_usage.py:161: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Creative Content Director
## Thought: I need to gather more specific information and insights related to the Inflation lookout in the Netherlands for Q4 2025 and 2026 from external sources to enrich the content creation process.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Inflation outlook in the Netherlands Q4 2025 2026\"}"
## Tool Output: 

Search results: Title: Economic forecast for Netherlands - Economy and Finance
Link: https://economy-finance.ec.europa.eu/economic-surveillance-eu-economies/netherlands/economic-forecast-netherlands_en
Snippet: Overall, annual HICP inflation is forecast at 3.0% in 2025 and 2.0% in 2026 .
---
Title: Netherlands Inflation Rate - Trading Economics
Link: https://tradingeconomics.com/netherlands/inflation-cpi
Snippet: In the long-term, the Netherlands Inflation Rate is projected to trend around 2.40 percent in 2026 and 2.50 percent in 2027, according to our econometric ...
---
Title: The state of the Dutch econom

/home/sacha/.cache/pypoetry/virtualenvs/datacamp-ml-M1zkPQRL-py3.10/lib/python3.10/site-packages/crewai/tools/tool_usage.py:161: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  acceptable_args = tool.args_schema.schema()["properties"].keys()  # type: ignore # Item "None" of "type[BaseModel] | None" has no attribute "schema"




# Agent: Creative Content Director
## Thought: Thought: I should search the internet for more detailed insights on the Inflation outlook in the Netherlands for Q4 2025 and 2026 to enhance the content creation process.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Inflation outlook in the Netherlands Q4 2025 2026 detailed analysis\"}"
## Tool Output: 

Search results: Title: Dutch household and government spending drives growth, increase ...
Link: https://www.rabobank.com/knowledge/d011468947-dutch-household-and-government-spending-drives-growth-increase-in-exports-expected-despite-geopolitical-turbulence
Snippet: Inflation is projected to be 3.6% in 2025 and 3.1% in 2026 , still well above the policy target of 2%. Wages are rising faster than inflation, improving purchasing power. We expect exports to rebound this year, growing by 2.4%. However, this prediction is surrounded by uncertainty due to the onset of the trade war.
---
Title: Economic forecast for 

## Social Content

In [37]:
import textwrap

posts = result.pydantic.model_dump()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
Analyzing the inflation outlook for the
Netherlands in Q4 2025 and 2026. Key economic
indicators, inflation drivers, and strategic
recommendations. #Netherlands #Economy #Inflation
--------------------------------------------------
LinkedIn
Insights into the economic indicators shaping the
inflation outlook for the Netherlands in Q4 2025
and 2026. A comprehensive analysis of GDP growth,
inflation trends, and unemployment rates.
#EconomicOutlook #Inflation #GDPGrowth
--------------------------------------------------


## Blog Post

In [36]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.dict()['article']))

/tmp/ipykernel_476891/3784749904.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  display(Markdown(result.pydantic.dict()['article']))


## Introduction
In analyzing the inflation outlook for the Netherlands in Q4 2025 and 2026, it is crucial to delve into key economic indicators and trends shaping the landscape of the Dutch economy. This period presents a dynamic environment influenced by various factors impacting inflation, GDP growth, employment rates, and government financial forecasts. Understanding the drivers behind these metrics is essential for investors, policymakers, and individuals alike to make informed decisions in a rapidly changing economic scenario.

## Economic Indicators Overview
### GDP Growth
GDP growth in the Netherlands for Q4 2025 and 2026 is anticipated to exhibit moderate expansion driven by robust domestic demand, stable exports, and ongoing investments in key sectors. However, uncertainties surrounding global trade dynamics and geopolitical tensions pose challenges to sustained growth momentum, necessitating a cautious approach to economic forecasting and policy formulation.

### Inflation Trends
Inflation trends are expected to fluctuate in the upcoming quarters, influenced by factors such as fluctuating energy prices, supply chain disruptions, and evolving consumer preferences. The impact of these dynamics on consumer purchasing power and overall price stability underscores the importance of proactive monitoring and adaptive strategies by businesses and policymakers to mitigate inflationary pressures.

### Unemployment Rates
Unemployment rates in the Netherlands are projected to gradually decline in Q4 2025 and 2026, reflecting a recovering labor market and increasing employment opportunities across various sectors. However, structural challenges and skill-mismatch issues necessitate targeted interventions to address disparities in workforce participation and promote inclusive growth strategies for long-term economic sustainability.

### Government Deficit Projections
Government deficit projections remain a critical aspect of fiscal policy planning, with a focus on balancing budgetary priorities, debt sustainability, and strategic investments for economic growth. As the government navigates through revenue challenges and expenditure priorities, transparent and accountable fiscal management practices are essential to maintain investor confidence and macroeconomic stability.

## Key Drivers of Inflation
### Wage Growth
Wage growth plays a pivotal role in shaping inflation dynamics, influencing consumer spending patterns, production costs, and overall price levels. Understanding the relationship between wage trends, labor market dynamics, and inflation expectations is essential for assessing the impact on business profitability, household incomes, and inflationary pressures in the Dutch economy.

### Services Inflation
Services inflation reflects the pricing dynamics of essential services such as healthcare, education, and utilities, affecting consumer affordability and cost of living standards. Analyzing the drivers behind services inflation, including regulatory changes, demand-supply imbalances, and technological advancements, provides valuable insights into inflationary trends and policy implications for service-oriented industries.

### Processed Food Costs
Processed food costs represent a significant component of consumer expenditure, with price fluctuations influenced by agricultural output, supply chain efficiencies, and market competition. Monitoring processed food costs and analyzing price movements helps anticipate inflationary trends, consumer behavior shifts, and policy responses to safeguard food security and affordability for households in the Netherlands.

### Energy Prices
Energy prices play a dual role in the economy, impacting production costs for businesses and households' energy expenses. Fluctuations in energy prices, driven by global market dynamics, environmental regulations, and technological innovations, have implications for inflation, industry competitiveness, and energy transition strategies in the context of sustainable economic development and climate resilience.

## Strategic Recommendations and Economic Outlook
In light of the economic indicators and inflation drivers discussed, strategic recommendations for investors and policymakers include prudent risk management, diversified investment portfolios, cost-effective pricing strategies, and targeted policy interventions to address inflationary pressures and macroeconomic vulnerabilities. Embracing data-driven analysis, sector-specific insights, and proactive risk monitoring can enhance decision-making capabilities and foster resilience in navigating the evolving economic landscape of the Netherlands for Q4 2025 and 2026.

In conclusion, the inflation outlook in the Netherlands for Q4 2025 and 2026 underscores the importance of informed decision-making, strategic foresight, and collaborative initiatives across public and private sectors to promote economic stability, sustainable growth, and inclusive prosperity amidst a complex and interconnected global economy. By leveraging actionable insights, embracing innovation, and fostering a culture of resilience, stakeholders can navigate challenges and capitalize on opportunities to steer the Dutch economy towards a prosperous and resilient future.

In [30]:
# Get the final answer
#print(result.final_output)

# Get the raw LLM responses
#print(result.raw)

# (if supported) Get per-task outputs
print(len(result.tasks_output))

4


In [31]:
# Convert to a plain dict
pydantic_dict = result.pydantic.dict()
print(pydantic_dict.keys())

# Or inspect Pydantic fields directly
print(result.pydantic.__fields__.keys())

dict_keys(['article', 'social_media_posts'])
dict_keys(['article', 'social_media_posts'])


/tmp/ipykernel_476891/3450835516.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  pydantic_dict = result.pydantic.dict()
/tmp/ipykernel_476891/3450835516.py:6: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(result.pydantic.__fields__.keys())
